In [1]:
import pandas as pd
import numpy as np

In [2]:
general_df = pd.read_csv('./generalizability/ldaMataveMetrics.csv')

In [3]:
general_df

,dataset,model,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,ZERO
0,yahoo,LDA,9.436303e-01,0.122554,0.888889,0.412986,1.012399,0.349915,0.103568,0.713425,0.374836,0.270910
1,yahoo,MATAVE,9.916591e-01,0.222275,0.962963,0.416155,1.028472,0.376042,0.096035,0.772034,0.562229,0.295235
2,yahoo,combinedTopicModel,9.995372e-01,0.158400,0.874558,0.414935,0.991504,0.214404,0.104635,0.720164,0.404874,0.252905
3,yahoo,realToReal,1.000000e+00,0.005036,0.416854,0.403372,0.880048,0.010016,0.029556,0.039246,0.148319,0.100081
4,yahoo,realToReal2,1.000000e+00,0.007992,0.593496,0.401987,0.865898,0.008375,0.012696,0.049415,0.155089,0.101293
5,yahoo,realToReal3,1.000000e+00,0.010083,0.373745,0.401627,0.868195,0.011692,0.014183,0.039774,0.177254,0.101734
6,banking77,LDA,2.776298e-13,0.131400,0.903989,0.310814,0.615170,0.382805,0.308852,0.703925,0.365555,0.147635
7,banking77,MATAVE,6.055634e-03,0.176033,0.991870,0.337082,0.716313,0.469054,0.637870,0.882104,0.445697,0.220855
8,banking77,combinedTopicModel,9.495060e-01,0.165787,0.934553,0.322378,0.623337,0.340998,0.442302,0.660813,0.396567,0.186324
9,banking77,realToReal,1.000000e+00,0.018521,0.609782,0.258443,0.365246,0.038515,0.007754,0.028652,0.154659,0.084253


In [4]:
calibrated_dfs = []
for dataset in general_df['dataset'].value_counts().keys().tolist():
    temp_df = general_df[general_df['dataset'] == dataset]
    temp_df = temp_df.drop(columns='dataset')

    # Get metric columns. 
    metric_cols = [c for c in temp_df.columns if 'model' not in c]
    # Extract real rows.
    real_mask = temp_df['model'].str.startswith('real')
    real_df = temp_df.loc[real_mask, metric_cols]

    # Get mean and standard deviation. 
    mean = real_df.mean()
    std_dev = real_df.std(ddof=1)
    # There should be a minimum standard deviation, otherwise the metric is returning all 0s or exhibits no variation (which we assume is not indicative of a perfect metric, but a failing one).
    min_std = 1e-4

    # Reliability is inverse standard deviation. 
    reliability = pd.Series(index=std_dev.index, dtype=float)

    for metric in std_dev.index:
        # Real-vs-real values for this metric
        real_values = real_df[metric]
        print(real_values)

        # Detect saturation at minimum or maximum
        all_zero = np.allclose(real_values, 0, atol=0.01)
        all_one = np.allclose(real_values, 1, atol=0.01)

        if all_zero or all_one:
            reliability[metric] = 0

        elif std_dev[metric] < min_std:
            reliability[metric] = 0

        else:
            reliability[metric] = 1 / std_dev[metric]

    if reliability.sum() == 0:
        raise ValueError(
            "All metrics were assigned zero reliability."
        )

    # Safe standard deviation for z-scoring.
    safe_std = std_dev.copy()
    safe_std[safe_std < min_std] = min_std


    weights = reliability / reliability.sum()
    weight_rank = weights.rank(ascending=False, method="dense").astype(int)
    
    # Calibrate scores.
    calibrated = temp_df.copy()

    for m in metric_cols:
        calibrated[m] = (
            (temp_df[m] - mean[m])
            / safe_std[m]
        )
        calibrated[f"rank_{m}"] = weight_rank[m]

    calibrated["final_score"] = (
        calibrated[metric_cols] * weights
    ).sum(axis=1)

    calibrated['dataset'] = dataset

    calibrated_dfs.append(calibrated)

3    1.0
4    1.0
5    1.0
Name: CHI, dtype: float64
3    0.005036
4    0.007992
5    0.010083
Name: ZIPF, dtype: float64
3    0.416854
4    0.593496
5    0.373745
Name: CLASSIFIER, dtype: float64
3    0.403372
4    0.401987
5    0.401627
Name: IRPR, dtype: float64
3    0.880048
4    0.865898
5    0.868195
Name: FID, dtype: float64
3    0.010016
4    0.008375
5    0.011692
Name: PR, dtype: float64
3    0.029556
4    0.012696
5    0.014183
Name: DC, dtype: float64
3    0.039246
4    0.049415
5    0.039774
Name: MAUVE, dtype: float64
3    0.148319
4    0.155089
5    0.177254
Name: TRADITIONAL, dtype: float64
3    0.100081
4    0.101293
5    0.101734
Name: ZERO, dtype: float64
9     1.000000
10    1.000000
11    0.999554
Name: CHI, dtype: float64
9     0.018521
10    0.015009
11    0.013292
Name: ZIPF, dtype: float64
9     0.609782
10    0.347368
11    0.490507
Name: CLASSIFIER, dtype: float64
9     0.258443
10    0.255469
11    0.262185
Name: IRPR, dtype: float64
9     0.365246
10    0.3

In [5]:
calibrated_results = pd.concat(calibrated_dfs)
calibrated_results

,model,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,...,rank_CLASSIFIER,rank_IRPR,rank_FID,rank_PR,rank_DC,rank_MAUVE,rank_TRADITIONAL,rank_ZERO,final_score,dataset
0,LDA,-563.697174,45.287521,3.671598,11.564781,18.570181,204.958881,9.079844,117.144949,14.180594,...,9,2,6,3,7,5,8,1,109.818583,yahoo
1,MATAVE,-83.408583,84.609292,4.307750,15.002829,20.686738,220.714097,8.272847,127.382847,26.562514,...,9,2,6,3,7,5,8,1,127.130506,yahoo
2,combinedTopicModel,-4.628214,59.422155,3.548525,13.679438,15.818661,123.243023,9.194088,118.322146,16.165381,...,9,2,6,3,7,5,8,1,92.169750,yahoo
3,realToReal,0.000000,-1.052020,-0.382261,1.132477,1.141413,-0.006836,1.151035,-0.622905,-0.786369,...,9,2,6,3,7,5,8,1,-0.102587,yahoo
4,realToReal2,0.000000,0.113771,1.134744,-0.370991,-0.721977,-0.996565,-0.655138,1.153469,-0.339085,...,9,2,6,3,7,5,8,1,-0.156367,yahoo
5,realToReal3,0.000000,0.938249,-0.752484,-0.761486,-0.419436,1.003400,-0.495896,-0.530564,1.125454,...,9,2,6,3,7,5,8,1,0.258954,yahoo
6,LDA,-3880.162110,43.444244,3.207581,15.487224,56.088133,31.321924,88.215020,128.074400,32.591137,...,9,3,5,8,4,6,7,1,50.279082,banking77
7,MATAVE,-3856.661772,60.190278,3.876452,23.293524,78.692933,39.131934,185.066338,162.015016,45.129464,...,9,3,5,8,4,6,7,1,81.226561,banking77
8,combinedTopicModel,-195.376544,56.346192,3.440207,18.923769,57.913300,27.536213,127.498001,119.862121,37.442921,...,9,3,5,8,4,6,7,1,61.508459,banking77
9,realToReal,0.577350,1.093215,0.968353,-0.076028,0.231627,0.145823,-0.417914,-0.556193,-0.404111,...,9,3,5,8,4,6,7,1,0.295052,banking77


In [6]:
calibrated_results.to_csv('./generalizability.csv')

In [7]:
ranked_dict_results = []
for dataset in calibrated_results['dataset'].unique():
    subset = calibrated_results[calibrated_results['dataset'] == dataset]
    # Extract not real rows.
    real_mask = subset['model'].str.startswith('real')
    subset = subset.loc[~real_mask]
    ranks = subset['final_score'].rank(
        ascending=True,
        method='dense'
    ).astype(int)
    ranked_dict_results.append(pd.DataFrame({
        'model': subset['model'].values,
        'dataset': dataset,
        'final_score': subset['final_score'].values,
        'rank': ranks.values
    }))

In [8]:
ranked_results_df = pd.concat(ranked_dict_results)
ranked_results_df

,model,dataset,final_score,rank
0,LDA,yahoo,109.818583,2
1,MATAVE,yahoo,127.130506,3
2,combinedTopicModel,yahoo,92.169750,1
0,LDA,banking77,50.279082,1
1,MATAVE,banking77,81.226561,3
2,combinedTopicModel,banking77,61.508459,2
0,LDA,medicalAbstracts,42.129073,2
1,MATAVE,medicalAbstracts,63.467240,3
2,combinedTopicModel,medicalAbstracts,40.358034,1
0,LDA,dementiaAudio,96.233720,1


In [9]:
ranked_results_df[ranked_results_df['rank'] == 1]['model'].value_counts()

model
combinedTopicModel    7
LDA                   2
MATAVE                1
Name: count, dtype: int64